# 02 Harmonisation

One rule per section, each with the literature that supports it. Status as of 2026-10-31 (`research.md`): **[V]** = DOI resolved, authors/title/year/venue confirmed (full texts not opened, the "means for us" readings are inferences from abstracts); **[S]** = seen in search results only; **[U]** = no supporting source found, a documented design choice.
Inputs: tables from `01_datasets_features` in `data/interim/notebooks/` (real when 01 ran with `REAL_DATA = True`, synthetic otherwise; the first cell prints which, and falls back to synthetic if 01 has not run). Demos that need data the tables do not hold (fractal DEM, toy lunar cycle, Dirichlet GLiM fractions) are synthetic and say so. Outputs go to `data/interim/notebooks/` only (on Colab `MyDrive/DoppelGrund/out`, the folder 01 writes; run 01 first, and upload `doppelgrund_colab.zip` to `MyDrive/DoppelGrund/` as described there).

In [ ]:
import sys, subprocess, importlib.util, zipfile, json
from pathlib import Path
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:  # same setup as notebook 01; 01's outputs are read from MyDrive/DoppelGrund/out
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/DoppelGrund")
    if importlib.util.find_spec("h3") is None: subprocess.run([sys.executable, "-m", "pip", "install", "-q", "h3"], check=True)
    if not Path("/content/DoppelGrund/pipeline").exists(): zipfile.ZipFile(ROOT / "doppelgrund_colab.zip").extractall("/content/DoppelGrund")
    sys.path.insert(0, "/content/DoppelGrund/pipeline")  # not `pip install -e`: pyproject pins would fight Colab's numpy/pandas
import numpy as np, pandas as pd
from scipy.stats import skew, ks_2samp
from sklearn.ensemble import HistGradientBoostingClassifier as HGB
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from doppelgrund.config import REPO, EXPECTED_FEATURES, load_profiles, load_crosswalk
from doppelgrund.fixtures.synthetic import make_synthetic, fit_scaler, LITH_COLS

IN = OUT = ROOT / "out" if IN_COLAB else REPO / "data/interim/notebooks"
prof = load_profiles(); SEED = prof.params["seed"]
try:
    T = {k: pd.read_parquet(IN / f"{k}.parquet") for k in ("earth_features", "moon_discs", "mars_discs", "targets")}
    MODE = (IN / "tables_source.txt").read_text() if (IN / "tables_source.txt").exists() else "unknown"
except FileNotFoundError:
    T = make_synthetic(SEED, n_hex=6000)  # offline fallback, same seed as notebook 01
    MODE = "synthetic (no 01 outputs found)"
print("tables from notebook 01:", MODE)
earth = T["earth_features"]; earth = earth[~earth["masked"]] if "masked" in earth else earth   # barren/low-NDVI hexes only (mask)
moon, mars = T["moon_discs"], T["mars_discs"]
BODIES = {"earth": earth, "moon": moon, "mars": mars}
FEATS = list(EXPECTED_FEATURES)
{k: len(v) for k, v in BODIES.items()}

## 1. Provenance: same quantity vs proxy

Features are split into **same quantity on all bodies** and **proxy that differs by body**; proxies are shown here and down-weighted by default (research change 3). Nothing in the proxy rows should be read as physically identical across bodies.

In [ ]:
SAME, PROXY = "same quantity", "PROXY (differs by body)"
PROV = pd.DataFrame([
 ("slope_p50_deg / p90 / frac<10", "terrain", SAME, "GLO30 area-mean to 470 m, Horn", "LDEM_64 ~474 m, Horn", "MOLA 463 m, Horn", "baseline 470 m on all; residual resampling bias is an uncertainty"),
 ("relief_std_m", "terrain", SAME, "std of elevation, same footprint", "same", "same", "baseline-dependent (Kreslavsky & Head 2000)"),
 ("lst_mean_K", "thermal", PROXY, "mean of Terra day/night", "mean of 24 h Diviner bins", "none (NaN)", "Earth: 2-time-point mean; Moon: full cycle"),
 ("diurnal_range_K", "thermal", SAME, "Terra 10:30 minus 22:30", "Diviner GCP bins around 10:30 minus those around 22:30 (3x3 0.5 deg bins)", "none (NaN)", "matched local time (section 3)"),
 ("annual_range_K", "thermal", SAME, "max-min monthly mean", "NaN", "NaN", "Earth-only in practice, fails coverage"),
 ("frac_below_0C", "cold_ice", PROXY, "fraction of 8-day composites", "fraction of local-time bins", "none (NaN)", "time-series vs diurnal-cycle fraction"),
 ("permafrost_prob", "cold_ice", PROXY, "Obu et al. 2018 (NH, north of 25 N)", "NaN", "NaN", "Earth-only"),
 ("aridity_index", "cold_ice", PROXY, "TerraClimate P/PET", "0 by definition", "0 by definition", "assumption in profiles.yaml"),
 ("lith_* fractions", "composition", PROXY, "GLiM via crosswalk", "Lunar Prospector GRS FeO (about 5 deg)", "Tanaka 2014 geologic unit via a PROPOSED mapping (01)", "coarse ontology; Hellinger (section 4)"),
], columns=["feature", "group", "kind", "earth", "moon", "mars", "note"])
PROV

## 2. Matched baseline and slope sensitivity

Report every slope at a stated baseline, use the same Horn algorithm everywhere, and area-mean (not nearest) when coarsening. Lunar slope changes strongly between 17 m and 1 km [V: Rosenburg et al. 2011, doi:10.1029/2010JE003716]; Mars roughness is defined at stated baselines [V: Kreslavsky & Head 2000, doi:10.1029/2000JE001259; V: Kreslavsky et al. 2013, doi:10.1016/j.icarus.2013.04.027 (Moon)]. Slopes from different baselines differ systematically (Mars MOLA/HRSC) [V: Wang & Wu 2017, doi:10.5194/isprs-archives-xlii-3-w1-155-2017]. Algorithm and resolution error [V: Zhou & Liu 2004, doi:10.14358/pers.70.8.957; V preprint, not peer reviewed: Grohmann 2017, doi:10.31223/osf.io/6xc29].
**Demo on a synthetic fractal DEM** (no remote data): 30 m surface, area-mean to 470 m (the stage-1 baseline) and to 240 m (second baseline, like LDEM_128). The ratio is a documented uncertainty, not a correction.

In [ ]:
def horn_slope_deg(z, dx):
    """Same Horn 3x3 operator as notebook 01 (square cells, dx metres)."""
    p = np.pad(z, 1, mode="edge"); a,b,c,d,f,g,h,i = (p[:-2,:-2],p[:-2,1:-1],p[:-2,2:],p[1:-1,:-2],p[1:-1,2:],p[2:,:-2],p[2:,1:-1],p[2:,2:])
    return np.degrees(np.arctan(np.hypot(((c+2*f+i)-(a+2*d+g))/(8*dx), ((g+2*h+i)-(a+2*b+c))/(8*dx))))[1:-1, 1:-1]

def fractal_dem(n, beta=2.4, relief=15.0, seed=1):
    f = np.hypot(*np.meshgrid(np.fft.fftfreq(n), np.fft.fftfreq(n))); f[0, 0] = 1
    z = np.fft.ifft2(np.fft.fft2(np.random.default_rng(seed).normal(size=(n, n))) * f ** (-beta / 2)).real
    return z / z.std() * relief

def block_mean(z, k): n = z.shape[0] // k * k; return z[:n, :n].reshape(n // k, k, n // k, k).mean((1, 3))

z30 = fractal_dem(2400)
rows = []
for name, k in [("30 m native", 1), ("240 m (second baseline)", 8), ("470 m (stage 1, 16x30=480)", 16)]:
    s = horn_slope_deg(block_mean(z30, k), 30.0 * k)
    rows.append(dict(baseline=name, p50=np.percentile(s, 50), p90=np.percentile(s, 90), frac_lt10=(s < 10).mean()))
SENS = pd.DataFrame(rows).set_index("baseline").round(3)
SENS.loc["ratio 470 / 240"] = (SENS.iloc[2] / SENS.iloc[1]).round(3)
SENS

## 3. Overpass-matched thermal

Moon at the Diviner GCP local-time bins around 10:30 and 22:30 (the 0.25 h bins are centred at 10.375 and 10.625 h, so 10:30 is the mean of those two, likewise 22.375 and 22.625; notebook 01 reduces the 18 GCP tables to this because the 0.5 deg derived GeoTIFF had lost local time), so `diurnal_range_K = T(10:30) - T(22:30)` matches MODIS **Terra** (10:30 / 22:30) [V: Williams et al. 2017, doi:10.1016/j.icarus.2016.08.012; V: Wan 2014, doi:10.1016/j.rse.2013.08.027, about 1 K LST accuracy on barren land]. The bin must be stated. Aqua (13:30 / 01:30) would give a second pair as a robustness check (research change 2, not computed). The full-cycle lunar range is a **separate flagged gap**, reported below in kelvin.
Mars thermal inertia is an apparent value tied to local time and season [V: Putzig & Mellon 2007, doi:10.1016/j.icarus.2007.05.013], so it stays Mars-only (NaN on Earth) unless an Earth equivalent from MODIS day/night pairs is built. Notebook 01 keeps TES thermal inertia only as an unscored side column `tes_ti` on the Mars discs and does **not** convert it to a temperature, so Mars thermal features are NaN (an honest gap).
**Demo**: an illustrative toy lunar surface-temperature cycle (not Diviner data; peak about 390 K x (cos lat cos(h-angle))^0.25, night 95 K).

In [ ]:
def toy_moon_cycle(lat, h):
    """ILLUSTRATIVE toy, not Diviner. h = local solar time in hours (12 = noon)."""
    mu = np.cos(np.radians(lat)) * np.cos(np.radians(15 * (h - 12)))
    return np.where(mu > 0, np.maximum(390 * np.maximum(mu, 0) ** 0.25, 95.0), 95.0)

h = np.arange(96) * 0.25; lats = np.array([0, 20, 40, 60, 80])
cyc = toy_moon_cycle(lats[:, None], h[None, :])
i_d, i_n = np.argmin(abs(h - 10.5)), np.argmin(abs(h - 22.5))
GAP = pd.DataFrame({"matched_10:30-22:30_K": cyc[:, i_d] - cyc[:, i_n], "full_cycle_max-min_K": cyc.max(1) - cyc.min(1)}, index=lats)
GAP["gap_K"] = GAP["full_cycle_max-min_K"] - GAP["matched_10:30-22:30_K"]
print("bins used: nearest 0.25 h to 10.5 h and 22.5 h =", h[i_d], h[i_n])
GAP.round(1).rename_axis("lat_deg")

In [ ]:
# what the loaded tables hold for this feature (synthetic moon values are from a different generator than the toy above)
pd.DataFrame({k: v["diurnal_range_K"].describe()[["count", "50%", "min", "max"]] for k, v in BODIES.items()}).round(1)

## 4. Lithology to the shared ontology (crosswalk)

GLiM has 16 first-level classes [V: Hartmann & Moosdorf 2012, doi:10.1029/2012GC004370]. Mapped to the six-class ontology, keeping unconsolidated and mixed classes at low weight (research change from section 3). ASTER GED enters as a continuous index, not a class [V: Hulley et al. 2015, doi:10.1002/2015GL065564]. Moon (Lunar Prospector FeO, about 5 deg resolution) and Mars (Tanaka 2014 geologic units mapped to the classes by a proposal in 01) measure different things from ASTER and GLiM, so the ontology stays coarse and the proxy used per body is reported [V: Crites et al. 2015, doi:10.2138/am-2015-4874; Ruff & Christensen 2002, doi:10.1029/2001JE001580].
`config/ontology_crosswalk.csv` is **empty** (human-owned), so notebook 01 uses a **proposed** GLiM mapping (and a proposed Mars unit mapping, both for human review). 01 writes the GLiM table to `glim_ontology_proposal.csv` next to its tables and the demo below reads it back, so there is one definition. Nothing is written to config.

In [ ]:
xw = load_crosswalk()
print("config crosswalk rows:", len(xw))
O = [x.removeprefix("lith_") for x in LITH_COLS]
if (IN / "glim_ontology_proposal.csv").exists():
    # GLiM first-level -> ontology fractions (rows sum to 1, or all 0 for nd/wb), PROPOSAL for human review, written by notebook 01 (the table it used for Earth)
    P = pd.read_csv(IN / "glim_ontology_proposal.csv", index_col="code"); XW, GLIM_W = P[O], P["weight"]  # weights: low for unconsolidated/mixed classes
    def glim_to_ontology(frac):
        """frac: DataFrame (cells x 16 GLiM classes) of area fractions -> (cells x 6) ontology fractions; unmapped (nd/wb) share is dropped and renormalised."""
        w = frac[list(XW.index)].mul(GLIM_W)
        out = w.to_numpy() @ XW.to_numpy(); s = out.sum(1, keepdims=True)
        return pd.DataFrame(np.divide(out, s, out=np.full_like(out, np.nan), where=s > 0), index=frac.index, columns=O)

    demo = pd.DataFrame(np.random.default_rng(SEED).dirichlet(np.ones(16) * 0.3, 5), columns=list(XW.index))  # SYNTHETIC GLiM fractions
    onto = glim_to_ontology(demo); assert np.allclose(onto.sum(axis=1).dropna(), 1)
    display(onto.round(2))
else:
    print("run notebook 01 first (any mode): it writes glim_ontology_proposal.csv; skipping the GLiM demo")

### Hellinger distance on class fractions

`H(p, q) = sqrt(0.5 * sum((sqrt p - sqrt q)^2))`, bounded in [0, 1], symmetric, finite with zeros. **[U] No peer-reviewed planetary precedent found** (general support: Hellinger transform for compositions [V: Legendre & Gallagher 2001, doi:10.1007/s004420100716]; Hellinger/Bhattacharyya distances [V: Kailath 1967, doi:10.1109/TCOM.1967.1089532]); the choice rests on those properties, so cite it as a design decision. Rows with NaN or zero total are skipped.

In [ ]:
def hellinger(P, Q):
    """Pairwise Hellinger distance between rows of P (n,6) and Q (m,6); NaN row -> NaN."""
    return np.sqrt(0.5 * ((np.sqrt(P)[:, None, :] - np.sqrt(Q)[None, :, :]) ** 2).sum(-1))
assert hellinger(np.eye(6)[:1], np.eye(6)[1:2])[0, 0] == 1.0
norm = lambda d: d[LITH_COLS].to_numpy(float) / d[LITH_COLS].sum(axis=1, skipna=False).to_numpy(float)[:, None]
E, Mo, Ma = (norm(d) for d in (earth, moon, mars)); rng = np.random.default_rng(SEED)
pick = lambda A: A[rng.choice(len(A), min(300, len(A)), replace=False)]  # random rows: disc ids run from the north pole southward
E3, Mo3, Ma3 = pick(E), pick(Mo), pick(Ma)
pd.DataFrame({"earth-moon": np.nanpercentile(hellinger(E3, Mo3), [5, 50, 95]),
              "earth-mars": np.nanpercentile(hellinger(E3, Ma3), [5, 50, 95])}, index=["p5", "p50", "p95"]).round(3)

## 5. log1p on skewed features

The same transform for all bodies (set by `config/profiles.yaml`: `relief_std_m` and `aridity_index`), so pooled statistics are comparable. No supporting literature (engineering choice, [U]). Skewness before/after on the pooled values:

In [ ]:
LOG = [f.name for f in prof.features if f.transform == "log1p"]
def tf(df):
    d = df[FEATS].astype("float64").copy()
    for c in LOG: d[c] = np.log1p(d[c].clip(lower=0))
    return d
TF = {k: tf(v) for k, v in BODIES.items()}
pool_raw = pd.concat([v[FEATS] for v in BODIES.values()]).astype(float); pool_tf = pd.concat(TF.values())
pd.DataFrame({"skew_raw": skew(pool_raw[LOG], nan_policy="omit"), "skew_log1p": skew(pool_tf[LOG], nan_policy="omit")}, index=LOG).round(2)

## 6. Pooled median/MAD scaling and tolerances s_f

Robust scale `s_f = 1.4826 * MAD` of the transformed feature, pooled over Earth + Moon + Mars finite values, via the existing `fit_scaler`. `s_f` doubles as the per-feature tolerance in the similarity kernel. **[U] no peer-reviewed geoscience source** for pooled-MAD scaling, a documented engineering choice (sklearn `RobustScaler` equivalent). Pooling means a feature NaN for one body is scaled by the others only, and body row counts are unequal (Earth hexes vs 3000 discs per body).

In [ ]:
SC = pd.DataFrame(fit_scaler([v for v in BODIES.values()], prof)).set_index("name")
Z = {k: (v - SC["median"]) / SC["s"] for k, v in TF.items()}
SC.round(3)

## 7. Pooled 0.5/99.5 percentile clipping (clustering only)

Clip the scaled values at the pooled 0.5 and 99.5 percentiles. Used for clustering/embedding only; **similarity scores use the unclipped z** so extreme-but-real terrain is not flattened. [U] engineering choice.

In [ ]:
pz = pd.concat(Z.values()); LO, HI = pz.quantile(0.005), pz.quantile(0.995)
ZC = {k: v.clip(LO, HI, axis=1) for k, v in Z.items()}
pd.DataFrame({"lo": LO, "hi": HI}).round(2)

## 8. NaN coverage and the 80% rule

A feature is **usable for a body** only if at least 80% of its rows are finite. Features failing the rule on a body are dropped from that body's comparisons (the pair weight is renormalised over the remaining features, with `profiles.params.coverage_floor` as the pair-level floor). A feature that is all NaN for a body (Mars thermal) has coverage 0, fails the rule and is skipped by the domain-gap tests below. NaN is kept as missing, never imputed (HistGradientBoosting handles NaN natively). Lithology coverage is counted as a row with a finite fraction vector.

In [ ]:
COV = pd.DataFrame({k: v[FEATS].notna().mean() for k, v in BODIES.items()})
COV.loc["lithology"] = [v[LITH_COLS].notna().all(axis=1).mean() for v in BODIES.values()]
USABLE = COV >= 0.80
print("usable features per body:", USABLE.sum().to_dict()); print("floor for pair coverage:", prof.params["coverage_floor"])
COV.round(2).assign(**{f"ok_{c}": USABLE[c] for c in COV.columns})

## 9. Domain-gap diagnostic

Classifier two-sample test: HistGradientBoosting, 5-fold cross-validated ROC AUC, Earth vs body [V: Lopez-Paz & Oquab 2017, arXiv:1610.06545]. Added per research: a **permutation p-value** (labels shuffled, 100 times here, 200 recommended) and **per-feature** univariate checks (single-feature AUC and KS) because a single AUC says shift exists, not whether it is harmful [V: Rabanser et al. 2019, arXiv:1810.11953; venue from memory]. No AUC threshold is hard-coded (the 0.7-0.75 rule of thumb comes from blogs [S]). Features that are all NaN in either body (Mars thermal) are skipped, and `n_features` says how many entered. Run on all features and on the **same-quantity terrain** features only; the first is expected to separate trivially because proxy and NaN patterns differ by body.

In [ ]:
def gap(a, b, cols, n_perm=100, n=1500, seed=SEED):
    cols = [x for x in cols if a[x].notna().any() and b[x].notna().any()]  # an all-NaN feature only encodes the missing source
    rng = np.random.default_rng(seed)
    X = pd.concat([a[cols].sample(min(n, len(a)), random_state=seed), b[cols].sample(min(n, len(b)), random_state=seed)]).to_numpy()
    y = np.r_[np.zeros(min(n, len(a))), np.ones(min(n, len(b)))].astype(int)
    cv = StratifiedKFold(5, shuffle=True, random_state=seed)
    auc = lambda yy: roc_auc_score(yy, cross_val_predict(HGB(max_iter=40, random_state=seed), X, yy, cv=cv, method="predict_proba")[:, 1])
    obs = auc(y); null = np.array([auc(rng.permutation(y)) for _ in range(n_perm)])
    return dict(auc=obs, null_mean=null.mean(), p=(1 + (null >= obs).sum()) / (n_perm + 1), n_features=len(cols))
TERR = FEATS[:4]
GAP_TAB = pd.DataFrame({(b, s): gap(ZC["earth"], ZC[b], c) for b in ("moon", "mars") for s, c in (("all features", list(FEATS)), ("terrain only", TERR))}).T.round(3)
GAP_TAB

In [ ]:
def uni(a, b, c):
    x, y = a[c].dropna(), b[c].dropna()
    if len(x) < 30 or len(y) < 30: return dict(auc=np.nan, ks=np.nan)
    return dict(auc=roc_auc_score(np.r_[np.zeros(len(x)), np.ones(len(y))], np.r_[x, y]), ks=ks_2samp(x, y).statistic)
pd.concat({b: pd.DataFrame({c: uni(Z["earth"], Z[b], c) for c in FEATS}).T for b in ("moon", "mars")}, axis=1).round(2)

## 10. Save harmonised tables and scaler

Writes `harmonised_<body>.parquet` (z = unclipped, `zc_` = clipped for clustering; NaN preserved) and `scaler.json` (transforms, pooled median, `s_f`, clip bounds, coverage, usable flags) to `data/interim/notebooks/`.

In [ ]:
for k, src in BODIES.items():
    key = "h3" if k == "earth" else "disc_id"
    pd.concat([src[[key]].reset_index(drop=True), Z[k].add_prefix("z_").reset_index(drop=True), ZC[k].add_prefix("zc_").reset_index(drop=True)], axis=1).astype({f"z_{c}": "float32" for c in FEATS} | {f"zc_{c}": "float32" for c in FEATS}).to_parquet(OUT / f"harmonised_{k}.parquet", index=False)
(OUT / "scaler.json").write_text(json.dumps({
    "features": [{"name": c, "transform": "log1p" if c in LOG else "none", "median": float(SC.loc[c, "median"]), "s": float(SC.loc[c, "s"]),
                  "clip_lo": float(LO[c]), "clip_hi": float(HI[c])} for c in FEATS],
    "coverage": COV.round(4).to_dict(), "usable": USABLE.to_dict(), "coverage_rule": 0.80, "seed": SEED}, indent=2))
sorted(p.name for p in OUT.iterdir())